# Top-K trade logs - 6-section evaluation

Full trade logs for the C1-selected top-10 combos on the 20% OOS
test partition. Wins shaded green, losses red.

Sections:
1. Individual trade logs (unfiltered)
2. Combined trade log (unfiltered)
3. Monte Carlo on combined (unfiltered)
4. Individual trade logs with ML#2 (V3 filter)
5. Combined ML#2 trade log (event-driven)
6. Monte Carlo on combined ML#2

In [ ]:
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation.composed_strategy_runner import run_strategy, load_test_bars

# Import the V3 phase-2 eval module (provides build_combo_trades_test,
# solo_metrics, simulate_portfolio). We load it by file path because it
# sits under scripts/evaluation/ which isn't a proper package.
_spec = importlib.util.spec_from_file_location(
    '_v3eval', REPO / 'scripts/evaluation/final_holdout_eval_v3_c1_fixed500.py'
)
v3eval = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(v3eval)

TOP_STRATEGIES_PATH = REPO / 'evaluation' / 'top_strategies.json'
STARTING_EQUITY = 50_000.0
SIZINGS = [('fixed_dollars_500', 1.0), ('fixed5_2500', 5.0)]


def metrics_from_pnl(pnl, years_span, start_equity=STARTING_EQUITY):
    '''Headline metrics for a trade-PnL series.

    Sharpe is annualized: sharpe = (mean/std) * sqrt(trades_per_year), where
    trades_per_year = len(pnl) / years_span. This matches the standard
    convention for per-trade PnL under an i.i.d. assumption and is consistent
    across Sections 1/2/4/5.
    '''
    p = np.asarray(pnl, dtype=float)
    n = len(p)
    if n == 0:
        return dict(n_trades=0, trades_per_year=0.0, win_rate=0.0,
                    total_pnl_dollars=0.0, total_return_pct=0.0,
                    sharpe_ratio=0.0, max_drawdown_pct=0.0,
                    max_drawdown_dollars=0.0)
    eq_full = np.concatenate([[start_equity], start_equity + np.cumsum(p)])
    peak = np.maximum.accumulate(eq_full)
    dd_d = float((peak - eq_full).max())
    dd_pct = float(np.nan_to_num((peak - eq_full) / peak, nan=0.0).max() * 100)
    total = float(p.sum())
    tpy = n / years_span if years_span > 0 else 0.0
    std = p.std(ddof=1) if n > 1 else 0.0
    sharpe = float(p.mean() / std * np.sqrt(tpy)) if std > 0 and tpy > 0 else 0.0
    return dict(n_trades=int(n),
                trades_per_year=round(tpy, 1),
                win_rate=round(float((p > 0).mean()), 4),
                total_pnl_dollars=round(total, 2),
                total_return_pct=round(total / start_equity * 100, 2),
                sharpe_ratio=round(sharpe, 4),
                max_drawdown_pct=round(dd_pct, 2),
                max_drawdown_dollars=round(dd_d, 2))


def monte_carlo(pnl, start_equity=STARTING_EQUITY, n_sims=10_000, seed=42):
    '''IID bootstrap MC on a PnL series. Returns DD percentiles + VaR/CVaR +
    risk-of-ruin (>=50% DD) + permutation-test p-value on win rate.'''
    rng = np.random.default_rng(seed)
    p = np.asarray(pnl, dtype=float)
    n = len(p)
    if n == 0:
        return {'n_sims': n_sims, 'n_trades': 0, 'note': 'empty'}
    # Bootstrap DD distribution
    idx = rng.integers(0, n, size=(n_sims, n))
    samples = p[idx]
    equity = start_equity + np.cumsum(samples, axis=1)
    equity = np.concatenate([np.full((n_sims, 1), start_equity), equity], axis=1)
    peak = np.maximum.accumulate(equity, axis=1)
    dd_pct = np.nan_to_num((peak - equity) / peak, nan=0.0).max(axis=1) * 100
    # VaR / CVaR at 5th percentile trade level
    var5 = float(np.percentile(p, 5))
    tail = p[p <= var5]
    cvar = float(tail.mean()) if len(tail) else var5
    # Permutation test on win rate (one-tailed vs break-even 1/(1+avg_rr))
    # Use a simple two-sided bootstrap CI on wr for notebook readability
    wr = float((p > 0).mean())
    boot_wr = (samples > 0).mean(axis=1)
    wr_ci = (float(np.percentile(boot_wr, 2.5)), float(np.percentile(boot_wr, 97.5)))
    return {
        'n_sims': n_sims, 'n_trades': int(n),
        'win_rate': round(wr, 4),
        'wr_ci_95': (round(wr_ci[0], 4), round(wr_ci[1], 4)),
        'dd_p50_pct': round(float(np.percentile(dd_pct, 50)), 2),
        'dd_p90_pct': round(float(np.percentile(dd_pct, 90)), 2),
        'dd_p95_pct': round(float(np.percentile(dd_pct, 95)), 2),
        'dd_p99_pct': round(float(np.percentile(dd_pct, 99)), 2),
        'dd_worst_pct': round(float(dd_pct.max()), 2),
        'var_5pct_trade': round(var5, 2),
        'cvar_5pct_trade': round(cvar, 2),
        'risk_of_ruin_50pct_dd': round(float((dd_pct >= 50.0).mean()), 4),
    }


In [ ]:
payload = json.loads(TOP_STRATEGIES_PATH.read_text())
strategies = payload['top']
print(f'Loaded {len(strategies)} strategies (min_trades={payload["min_trades"]}, '
      f'eligible={payload["pool_sizes"]["eligible_combos"]:,}/'
      f'{payload["pool_sizes"]["total_combos"]:,})')
bars = load_test_bars()
print(f'Test partition: {len(bars):,} bars  {bars["time"].iloc[0]} -> {bars["time"].iloc[-1]}')

YEARS_SPAN = (pd.to_datetime(bars['time'].iloc[-1]) -
              pd.to_datetime(bars['time'].iloc[0])).total_seconds() / (365.25 * 86400)
print(f'Years span: {YEARS_SPAN:.3f}  (used to annualize Sharpe)')

print('\nRunning unfiltered composed_strategy_runner for each combo...')
results_raw = []
for s in strategies:
    print(f'  {s["global_combo_id"]}...', flush=True)
    results_raw.append(run_strategy(s, bars=bars))
print('Done (unfiltered).')

In [ ]:
import lightgbm as lgb
print('Loading V3 booster + calibrators...')
_v3inf = v3eval.v3inf
booster = lgb.Booster(model_file=str(_v3inf.V3_BOOSTER))
simple_cals = _v3inf._load_calibrators()
two_stage = _v3inf._load_per_combo_calibrators()

print('\nBuilding V3-filtered trades per combo (may take a few minutes)...')
combo_ids = [s['global_combo_id'] for s in strategies]
combos_ml2 = []
for gcid in combo_ids:
    print(f'  {gcid}...', flush=True)
    try:
        c = v3eval.build_combo_trades_test(gcid, booster, simple_cals, two_stage)
        print(f'    n_trades={c.get("n_trades", 0)}  rr={c.get("rr", float("nan")):.2f}')
    except Exception as e:
        c = {'combo_id': gcid, 'error': str(e)}
        print(f'    ERROR: {e}')
    combos_ml2.append(c)
print('Done (ML2).')

In [ ]:
_EXIT_MAP = {'sl': 'SL', 'tp': 'TP', 'time': 'EOD', 'force': 'EOD'}

def render_trade_log(df, columns=None):
    default = ['combo_id', 'date', 'direction', 'entry_px', 'sl_px', 'tp_px',
               'contracts', 'dollar_risk', 'dollar_reward', 'exit_px',
               'exit_reason', 'actual_pnl']
    cols = [c for c in (columns or default) if c in df.columns]
    styled = df[cols].copy()
    if 'date' in styled.columns and pd.api.types.is_datetime64_any_dtype(styled['date']):
        styled['date'] = styled['date'].dt.strftime('%Y-%m-%d %H:%M')
    if 'exit_reason' in styled.columns:
        styled['exit_reason'] = (styled['exit_reason'].astype(str).str.lower()
                                 .map(_EXIT_MAP).fillna(styled['exit_reason']))
    fmt = {}
    for col in ['entry_px', 'sl_px', 'tp_px', 'dollar_risk', 'dollar_reward',
                'exit_px', 'actual_pnl']:
        if col in styled.columns:
            fmt[col] = '{:.2f}'
    if 'contracts' in styled.columns:
        fmt['contracts'] = '{:.0f}'
    def colour(row):
        n = len(row)
        v = row.get('actual_pnl')
        if pd.notna(v) and str(v).strip() not in ('', 'nan'):
            try:
                return (['background-color: #c8e6c9; color: #1b5e20'] * n
                        if float(v) > 0
                        else ['background-color: #ffcdd2; color: #b71c1c'] * n)
            except (ValueError, TypeError):
                pass
        return [''] * n
    try:
        from IPython.display import display
        display(styled.style.apply(colour, axis=1).format(fmt, na_rep='-').hide(axis='index'))
    except Exception:
        print(styled.to_string(index=False))

## 1) Individual trade logs (unfiltered)

In [ ]:
from IPython.display import display, Markdown
for r in results_raw:
    display(Markdown(f"### {r['combo_id']} - {len(r['trades'])} trades"))
    if r['trades'].empty:
        print('(no trades)')
    else:
        t = r['trades'].copy(); t.insert(0, 'combo_id', r['combo_id'])
        render_trade_log(t)

## 2) Combined trade log (unfiltered)

In [ ]:
frames = []
for r in results_raw:
    if r['trades'].empty: continue
    t = r['trades'].copy(); t.insert(0, 'combo_id', r['combo_id'])
    frames.append(t)
combined_raw = (pd.concat(frames, ignore_index=True)
                .sort_values('date', kind='mergesort')
                .reset_index(drop=True)
                if frames else pd.DataFrame())
display(Markdown(f"### Combined unfiltered - {len(combined_raw):,} trades across "
                 f"{combined_raw['combo_id'].nunique() if not combined_raw.empty else 0} combos"))
if combined_raw.empty:
    print('(no trades)')
else:
    render_trade_log(combined_raw)

## 3) Monte Carlo on combined (unfiltered)

In [ ]:
rows = []
if not combined_raw.empty:
    for label, scale in SIZINGS:
        pnl = combined_raw['actual_pnl'].to_numpy() * scale
        rows.append({'sizing': label, **monte_carlo(pnl)})
mc_raw = pd.DataFrame(rows)
mc_raw

## 4) Individual trade logs with ML#2 (V3 filter, fixed $500)

In [ ]:
# Build V3-filtered trade logs per combo (event-driven, fixed $500).
def _shape_ml2_trades(c, fixed_dollars):
    if c.get('error') or c.get('n_trades', 0) == 0:
        return pd.DataFrame()
    rows = []
    for ti in range(c['n_trades']):
        contracts = int(fixed_dollars // (c['sl_pts'][ti] * v3eval.DOLLARS_PER_POINT))
        if contracts <= 0:
            continue
        pnl = c['pnl_pts'][ti] * contracts * v3eval.DOLLARS_PER_POINT
        rows.append({
            'combo_id': c['combo_id'],
            'date': pd.to_datetime(c['entry_time'][ti]),
            'contracts': contracts,
            'dollar_risk': fixed_dollars,
            'pnl_pts': float(c['pnl_pts'][ti]),
            'actual_pnl': pnl,
            'label_win': int(c['label_win'][ti]),
            'pwin_simple': float(c['pwin_simple'][ti]),
        })
    return pd.DataFrame(rows)

for c in combos_ml2:
    trades_500 = _shape_ml2_trades(c, 500.0)
    display(Markdown(f"### {c.get('combo_id')} - {len(trades_500)} trades (V3-filtered, $500 sizing)"))
    if trades_500.empty:
        print(c.get('error', '(no trades)'))
    else:
        render_trade_log(trades_500, columns=['combo_id', 'date', 'contracts',
                                              'dollar_risk', 'pnl_pts',
                                              'label_win', 'pwin_simple',
                                              'actual_pnl'])

## 5) Combined ML#2 trade log (fixed $500)

In [ ]:
# Event-driven combined trade log (fixed $500).
def _sim_trades(combos, fixed_dollars):
    events = []
    for ci, c in enumerate(combos):
        if c.get('error') or c.get('n_trades', 0) == 0:
            continue
        for ti in range(c['n_trades']):
            events.append((int(c['entry_bar'][ti]), 0, ci, ti))
            events.append((int(c['exit_bar'][ti]), 1, ci, ti))
    events.sort(key=lambda e: (e[0], -e[1]))
    open_pos = {}
    rows = []
    for bar, kind, ci, ti in events:
        c = combos[ci]
        if kind == 0:
            contracts = int(fixed_dollars // (c['sl_pts'][ti] * v3eval.DOLLARS_PER_POINT))
            if contracts <= 0: continue
            open_pos[(ci, ti)] = contracts
        else:
            key = (ci, ti)
            if key not in open_pos: continue
            contracts = open_pos.pop(key)
            pnl = c['pnl_pts'][ti] * contracts * v3eval.DOLLARS_PER_POINT
            rows.append({
                'combo_id': c['combo_id'],
                'date': pd.to_datetime(bars['time'].iloc[bar]) if bar < len(bars) else pd.NaT,
                'contracts': contracts,
                'dollar_risk': fixed_dollars,
                'pnl_pts': float(c['pnl_pts'][ti]),
                'actual_pnl': pnl,
                'label_win': int(c['label_win'][ti]),
                'pwin_simple': float(c['pwin_simple'][ti]),
            })
    return pd.DataFrame(rows)

combined_ml2 = _sim_trades(combos_ml2, 500.0)
display(Markdown(f"### Combined ML2 portfolio - {len(combined_ml2):,} trades "
                 f"(fixed $500, event-driven bar-overlap arbitration)"))
if combined_ml2.empty:
    print('(no trades)')
else:
    render_trade_log(combined_ml2, columns=['combo_id', 'date', 'contracts',
                                            'dollar_risk', 'pnl_pts',
                                            'label_win', 'pwin_simple',
                                            'actual_pnl'])

## 6) Monte Carlo on combined ML#2

In [ ]:
rows = []
if not combined_ml2.empty:
    for label, scale in SIZINGS:
        pnl = combined_ml2['actual_pnl'].to_numpy() * scale
        rows.append({'sizing': label, **monte_carlo(pnl)})
mc_ml2 = pd.DataFrame(rows)
mc_ml2